# SI4006 · M5 · Amparo — **Cuantizar el VLM** (S15, Parte A)

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 5

---

Sigue el lab de la profe (`sesiones/s15/S15_Lab_Despliegue_RESUELTO.ipynb`, **Parte A**): el mismo modelo cargado dos veces,
**fp16** y **int4** (bitsandbytes, nf4), y una tabla antes/después con memoria, latencia p50/p95 y calidad con temperatura 0.

- **Qué se cuantiza:** el VLM **Qwen2-VL-2B**, el único modelo que corre en la GPU del sistema que se despliega (M3 razona por Groq).
  Por qué este y no otro: `m5/README.md`, sección *Decisión*.
- **Calidad:** el harness de M4 (11 casos: `campos`, `valvula`, `exacto`, `juez_vlm`).
  - El juez de `juez_vlm` es siempre el VLM en fp16.
  - M3 comparte su caché entre las dos cargas: si int4 lee lo mismo, M3 responde lo mismo, y un cambio solo puede venir de la lectura.
- Al final se **precalcula el índice** de M3 para el Space (tabla de despliegue de la profe).

La lógica vive en `amparo/pipelines/m5_cuantizacion.py`; este notebook solo la llama.

> GPU (T4, L4 o A100) y `GROQ_API_KEY` en los secretos de Colab.

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.45" accelerate bitsandbytes qwen-vl-utils pillow pandas groq sentence-transformers
# Colab trae torchao 0.10 y algunas librerias exigen >= 0.16 si lo encuentran; aqui no se usa.
%pip uninstall -q -y torchao
print('Listo.')

In [ ]:
# Clona el repo, o lo ACTUALIZA si ya estaba (la vez pasada la copia vieja corrio codigo viejo).
import os, sys, subprocess
from pathlib import Path

REPO_URL, RAMA = 'https://github.com/TomasPosada0626/Amparo.git', 'm5'
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'amparo' / 'nucleo' / 'schema.py').exists()), None)
if RAIZ is None:
    if Path('Amparo').is_dir():
        subprocess.run(['git', '-C', 'Amparo', 'pull', '-q', 'origin', RAMA], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '-b', RAMA, REPO_URL, 'Amparo'], check=True)
    RAIZ = Path('Amparo').resolve()
sys.path.insert(0, str(RAIZ))
print(subprocess.run(['git', '-C', str(RAIZ), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/amparo_m5_cuantizacion'
except ImportError:
    CARPETA = None

from amparo.pipelines import m5_cuantizacion as m5q

## 1 · El contrato, antes de medir

Como en S14: *"si no lo escriben antes, después cualquier resultado parece bueno"*.
- **Latencia:** en GPU, int4 con bitsandbytes suele ir **igual o más lento** que fp16, porque descomprime los pesos en cada paso. Lo que se gana es **memoria**, así que la cláusula de latencia solo pide que no empeore más de un 30 %.
- **La válvula (m10)** no se negocia: el VLM no puede inventar la fecha ni el plazo que no se ven.

In [ ]:
ctx = m5q.preparar(CARPETA)          # raiz, carpeta, clave de Groq, contrato y manifiesto (commit, GPU, versiones)

## 2 · M3, una sola vez para las dos cargas

In [ ]:
m5q.armar_m3(ctx);

## 3 · Parte A · fp16 y int4, en la misma GPU

`cargar(cuantizar)` y `latencias()` son las del lab.
- **Memoria:** lo que reservó cada carga en la GPU.
- **Latencia:** 5 lecturas de la imagen de m01, más una de calentamiento que no cuenta.
- **Calidad:** el harness de M4 completo.

Primero se carga fp16, que se queda cargado como juez; luego int4.

In [ ]:
medidas = m5q.medir(ctx)

## 4 · La tabla antes/después y el contrato

In [ ]:
df = m5q.tabla(ctx)
display(df)

### Lo que int4 leyó distinto

Campo por campo. Fechas, números y plazos son la debilidad conocida del VLM (README de M4): ahí es donde menos bits pueden pegar primero.

In [ ]:
distintas = m5q.lecturas_distintas(ctx)

In [ ]:
m5q.guardar(ctx)      # m5/cuantizacion.json en esta copia del repo y en Drive (entrega/)
m5q.descargar(ctx)    # baja el archivo: la copia de Colab se borra al cerrar

## 5 · Para el despliegue: el índice de M3 precalculado

La tabla de despliegue de la profe pide *"precalculen el índice de embeddings; que no se construya al arrancar"*.
Se calcula una vez aquí y el archivo va al repo (`m5/space/indice_m3.npz`). El Space solo lo carga.

In [ ]:
from amparo import despliegue as dsp
ruta = dsp.precalcular_indice()
import shutil; shutil.copy(ruta, Path(ctx.carpeta) / 'entrega' / ruta.name)
try:
    from google.colab import files; files.download(str(ruta))
except ImportError:
    pass

## Lo que va a M5

- **La tabla antes/después** y el veredicto del contrato: `m5/cuantizacion.json` y el README de M5.
- **Si el contrato se cumple**, la demo en vivo (Kaggle T4) usa el VLM en int4 (`AMPARO_VLM=auto`). Si no se cumple, va en fp16 (`AMPARO_VLM=fp16`).
- **`m5/space/indice_m3.npz`** va al repo para el Space.

*Amparo · M5 · S15 · Parte A.*